[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-01-duckdb-fundamentals.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · DuckDB Fundamentals: In-Process OLAP Explained
**certified-journeys / duckdb-certified** · Day 1 · Foundations

> **Goal for today:** Understand what makes DuckDB unique as an in-process OLAP engine, connect to it from Python, run your first queries, and measure its speed advantage over SQLite on analytical workloads.


In [ ]:
%pip install -q duckdb


## Step 1 · What Is DuckDB and Why Does It Exist?

DuckDB is an **embedded analytical database** — it runs entirely inside your Python process, with zero external servers or daemons.

| Feature | SQLite | DuckDB |
|---|---|---|
| Execution model | Row-at-a-time | Vectorised (columnar batches) |
| Target workload | OLTP (many small writes) | OLAP (few large scans) |
| Deployment | Embedded | Embedded |
| File format | `.db` | `.db` / `.duckdb` |
| Parallelism | Single-threaded reads | Multi-threaded by default |
| Direct file queries | No | Yes — CSV, Parquet, JSON |

**Vectorised execution** means DuckDB processes thousands of values in a single CPU instruction (SIMD), which is why aggregations over millions of rows feel instant.

### Architecture in one paragraph
DuckDB's storage layer uses **columnar compression** (run-length, bitpacking, dictionary). The query engine applies **predicate pushdown** and **late materialisation** — it only reads the columns a query touches. The result: full table scans complete in milliseconds that would take seconds in a row-store.

> **Reading:** [Why DuckDB](https://duckdb.org/why_duckdb) · [Architecture Internals](https://duckdb.org/docs/internals/overview)


In [ ]:
import duckdb

# Verify the installed version
print("DuckDB version:", duckdb.__version__)

# Connect() with no arguments → pure in-memory database.
# Nothing is written to disk; perfect for exploration.
con = duckdb.connect()

# The simplest possible query
result = con.execute("SELECT 42 AS answer").fetchone()
print("Answer:", result[0])  # → 42


### What just happened?
- **`duckdb.connect()`** with no path creates a fully functional in-memory database — no file is touched.
- **`con.execute(sql)`** returns a `DuckDBPyRelation`; `.fetchone()` retrieves a single row as a tuple.
- The import itself is the only setup step — **no server to start, no port to configure**.
- DuckDB's Python package bundles the entire engine (~30 MB wheel); there are no external shared-library dependencies.


## Step 2 · Connection Modes: In-Memory vs. Persistent

DuckDB offers two connection modes:

| Mode | How to open | Use case |
|---|---|---|
| In-memory | `duckdb.connect()` | Notebooks, ephemeral analysis |
| Persistent | `duckdb.connect('path/to/file.duckdb')` | Production pipelines, shared data |

**Important:** Only **one write connection** can hold a persistent `.duckdb` file at a time. Multiple read-only connections are fine: `duckdb.connect('file.duckdb', read_only=True)`.

The default connection shorthand `duckdb.sql(...)` uses a module-level in-memory connection — convenient for quick exploration but not recommended in multi-file projects (namespace collisions).


In [ ]:
import tempfile
import os

# Create a temporary directory for the persistent file
tmpdir = tempfile.mkdtemp()
db_path = os.path.join(tmpdir, "demo.duckdb")

# --- Persistent connection ---
con_p = duckdb.connect(db_path)
con_p.execute("CREATE TABLE IF NOT EXISTS greetings (msg VARCHAR)")
con_p.execute("INSERT INTO greetings VALUES ('Hello, DuckDB!')")
con_p.close()  # flush and release the file lock

# --- Re-open and read back ---
con_p2 = duckdb.connect(db_path)
rows = con_p2.execute("SELECT * FROM greetings").fetchall()
print("Persisted rows:", rows)   # [('Hello, DuckDB!',)]
con_p2.close()

# --- Read-only connection (safe for concurrent readers) ---
con_ro = duckdb.connect(db_path, read_only=True)
count = con_ro.execute("SELECT count(*) FROM greetings").fetchone()[0]
print("Row count (read-only):", count)
con_ro.close()


### What just happened?
- A `.duckdb` file was created on disk, written to, closed, and re-opened — demonstrating full durability.
- **`con.close()`** releases the write lock so other processes (or your next cell) can open the file.
- **`read_only=True`** is essential when multiple scripts or Jupyter kernels need to query the same file simultaneously.
- The file format is the same regardless of how many tables are stored — one file, one database.


## Step 3 · Generating Synthetic Data and Running Aggregations

DuckDB has a built-in **`range()`** table function and supports **`generate_series()`** — no need to create test data in Python first.

DuckDB SQL is a strict superset of PostgreSQL syntax with several analytical extensions:
- **`SUMMARIZE`** — quick summary statistics for any table or query
- **`DESCRIBE`** — schema inspection
- **List comprehensions**, **STRUCT literals**, **`ARRAY_AGG`** and more

> **Reading:** [DuckDB SQL Introduction](https://duckdb.org/docs/sql/introduction)


In [ ]:
import duckdb

con = duckdb.connect()  # fresh in-memory connection

# Generate a 1 million row synthetic sales table using DuckDB's built-in functions.
# random() returns a float in [0, 1); cast to create realistic ranges.
con.execute("""
CREATE TABLE sales AS
SELECT
    i                                           AS id,
    date '2023-01-01' + INTERVAL (i % 365) DAY  AS sale_date,
    CASE (i % 4)
        WHEN 0 THEN 'North'
        WHEN 1 THEN 'South'
        WHEN 2 THEN 'East'
        ELSE 'West'
    END                                         AS region,
    round(random() * 990 + 10, 2)               AS amount
FROM range(1, 1_000_001) t(i)
""")

# Verify row count
n = con.execute("SELECT count(*) FROM sales").fetchone()[0]
print(f"Rows: {n:,}")  # 1,000,000

# SUMMARIZE is a DuckDB extension — shows min/max/mean/null count in one shot
summary = con.execute("SUMMARIZE sales").df()
print(summary[["column_name", "min", "max", "approx_unique"]])


### What just happened?
- **`range(1, 1_000_001)`** is a DuckDB table function that streams integers — no intermediate Python list.
- **`CREATE TABLE AS SELECT`** (CTAS) materialises the result into a columnar in-memory table.
- **`.df()`** returns a Pandas DataFrame — DuckDB has zero-copy integration with both pandas and PyArrow.
- **`SUMMARIZE`** is DuckDB-specific; PostgreSQL doesn't have it. It's the fastest way to audit a new dataset.


## Step 4 · DuckDB vs SQLite: A Performance Comparison

SQLite is optimised for **OLTP** (transactional row writes/reads). DuckDB is optimised for **OLAP** (analytical scans over many rows).

The key implementation difference:
- SQLite reads one **row** at a time, evaluates every column even if not selected
- DuckDB reads one **column** at a time in vectorised batches of ~2,048 values

On a `SUM(amount)` over 1M rows, DuckDB reads only the `amount` column (a fraction of total data). SQLite reads every column in every row.

| Operation | SQLite strength | DuckDB strength |
|---|---|---|
| INSERT 1 row | ✓ fast | Comparable |
| SELECT with index | ✓ fast | Comparable |
| `SUM`/`AVG` over 1M rows | Slow (row scan) | ✓ Fast (columnar) |
| `GROUP BY` 1M rows | Slow | ✓ Fast |
| Multi-column aggregation | Very slow | ✓ Fast |


In [ ]:
import sqlite3
import time

ROWS = 1_000_000

# ── DuckDB benchmark ─────────────────────────────────────────────────────────
# The 'sales' table from Step 3 is already in memory (con is still open).
# Warm-up run to let DuckDB JIT-compile the query plan.
con.execute("SELECT region, SUM(amount) FROM sales GROUP BY region").fetchall()

start = time.perf_counter()
duck_result = con.execute(
    "SELECT region, SUM(amount), COUNT(*) FROM sales GROUP BY region ORDER BY region"
).fetchall()
duck_ms = (time.perf_counter() - start) * 1000
print(f"DuckDB  : {duck_ms:6.1f} ms")
for row in duck_result:
    print(f"  {row[0]:6s}  total={row[1]:>12,.0f}  count={row[2]:,}")

# ── SQLite benchmark ─────────────────────────────────────────────────────────
# Create an in-memory SQLite DB and fill it with the same 1M rows.
# This may take ~10–15 s because SQLite INSERT is row-at-a-time.
print("\nBuilding SQLite table (this takes a moment)…")
sq = sqlite3.connect(":memory:")
sq.execute("CREATE TABLE sales (id INT, sale_date TEXT, region TEXT, amount REAL)")

# Batch insert via executemany for speed
import random, datetime
regions = ["North", "South", "East", "West"]
base = datetime.date(2023, 1, 1)
batch = [
    (i, str(base + datetime.timedelta(days=i % 365)),
     regions[i % 4], round(random.random() * 990 + 10, 2))
    for i in range(1, ROWS + 1)
]
sq.executemany("INSERT INTO sales VALUES (?,?,?,?)", batch)

# Warm up
sq.execute("SELECT region, SUM(amount) FROM sales GROUP BY region").fetchall()

start = time.perf_counter()
sq.execute("SELECT region, SUM(amount), COUNT(*) FROM sales GROUP BY region ORDER BY region").fetchall()
sqlite_ms = (time.perf_counter() - start) * 1000
print(f"SQLite  : {sqlite_ms:6.1f} ms")

speedup = sqlite_ms / duck_ms
print(f"\nDuckDB is {speedup:.1f}x faster on this GROUP BY aggregation.")
sq.close()


### What just happened?
- DuckDB's columnar engine reads only the `region` and `amount` columns, skipping `id` and `sale_date` entirely.
- **Vectorised execution** processes ~2,048 values per CPU instruction; SQLite evaluates one row per iteration.
- Typical speedups on `GROUP BY` aggregations over 1M rows: **5–30×**, depending on the query shape.
- The Python batch insert into SQLite was itself ~10 s — DuckDB's `CREATE TABLE AS SELECT` from `range()` took milliseconds because it never left the database engine.


## Step 5 · Exploring DuckDB's SQL Dialect

DuckDB targets **PostgreSQL compatibility** but adds many analytical extensions that PostgreSQL lacks:

| Extension | Example |
|---|---|
| `EXCLUDE` columns | `SELECT * EXCLUDE (id, created_at) FROM t` |
| `REPLACE` columns | `SELECT * REPLACE (amount * 1.1 AS amount) FROM t` |
| `QUALIFY` (window filter) | `SELECT * FROM t QUALIFY row_number() OVER (PARTITION BY region ORDER BY amount DESC) = 1` |
| `PIVOT` / `UNPIVOT` | Native SQL pivot — no `CASE WHEN` gymnastics |
| Positional `GROUP BY` | `GROUP BY 1, 2` (same as `GROUP BY region, sale_date`) |
| Integer division | `5 // 2 = 2` |
| List comprehensions | `[x * 2 FOR x IN list_col IF x > 0]` |

> **Reading:** [DuckDB SQL Introduction](https://duckdb.org/docs/sql/introduction)


In [ ]:
# Demonstrate DuckDB-specific SQL extensions on the 'sales' table

# 1. SELECT * EXCLUDE — drop columns you don't need without listing all others
print("=== EXCLUDE ===")
df_excl = con.execute("SELECT * EXCLUDE (id) FROM sales LIMIT 3").df()
print(df_excl.to_string(index=False))

# 2. REPLACE — override one column in a SELECT * without repeating all column names
print("\n=== REPLACE ===")
df_repl = con.execute("""
    SELECT * REPLACE (round(amount * 1.1, 2) AS amount)
    FROM sales
    LIMIT 3
""").df()
print(df_repl.to_string(index=False))

# 3. Positional GROUP BY and ORDER BY
print("\n=== Positional GROUP BY ===")
rows = con.execute("""
    SELECT region, COUNT(*) AS n, round(AVG(amount), 2) AS avg_amount
    FROM sales
    GROUP BY 1          -- same as GROUP BY region
    ORDER BY 3 DESC     -- order by avg_amount descending
""").fetchall()
for r in rows:
    print(f"  {r[0]:6s}  n={r[1]:,}  avg=${r[2]}")

# 4. DESCRIBE — quick schema inspection
print("\n=== DESCRIBE ===")
schema = con.execute("DESCRIBE sales").fetchall()
for col in schema:
    print(f"  {col[0]:12s} {col[1]}")


### What just happened?
- **`SELECT * EXCLUDE`** is a major ergonomics win when tables have 20+ columns and you only want to drop 1–2.
- **`SELECT * REPLACE`** lets you apply transformations inline without retyping every column name.
- **Positional references** (`GROUP BY 1`) match the SELECT list by position — they match PostgreSQL behaviour.
- **`DESCRIBE`** returns column names, types, nullability, and defaults — useful for schema discovery on large files.


In [ ]:
# Challenge: Explore DuckDB's SQL dialect
#
# Using the 'sales' table already in `con`, write a single query that:
#   1. Filters to rows where amount > 500
#   2. Uses SELECT * EXCLUDE to drop the 'id' column
#   3. Adds a new column: revenue_tier = 'high' if amount >= 800, else 'mid'
#   4. Groups by region and revenue_tier, counts rows and sums amount
#   5. Orders by region ASC, total_amount DESC
#
# Expected columns in output: region, revenue_tier, cnt, total_amount

# Your solution here
# result = con.execute("""
#     SELECT ...
# """).df()
# print(result)


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| In-process engine | No server, no daemon — DuckDB runs inside your Python process |
| Columnar storage | Data stored column-by-column; queries touch only referenced columns |
| Vectorised execution | Processes ~2,048 values per CPU instruction (SIMD) |
| Connection modes | `duckdb.connect()` → in-memory; `duckdb.connect('file.duckdb')` → persistent |
| SQL extensions | `EXCLUDE`, `REPLACE`, `QUALIFY`, `SUMMARIZE`, positional `GROUP BY` |
| OLAP vs OLTP | DuckDB excels at scans/aggregations; SQLite excels at row-level transactions |

> **Tip:** DuckDB runs entirely in-process — no server to start, no daemon to manage. It's SQLite for analytics: embed it anywhere, query anything.

---
## What's next
**Day 2** → Reading files directly — query CSV, Parquet, and JSON without loading them into tables first.

Mark Day 1 complete in your [tracker](../index.html).
